# Story C: Behavioral Weirdness — Anomaly Detection

**Module:** Story C | **Methods:** Isolation Forest, Local Outlier Factor (LOF), Robust Z-Score  
**Author:** Vĩnh Hoàng | **Course:** Data Mining — MSA30DN, FSB  
**Instructor:** PhD. Cao Vu BUI

---

## Mục tiêu nghiên cứu

Xác định các **người dùng bất thường** (unusual users) và **phim gây phân cực** (polarizing movies) trong dữ liệu MovieLens.

**Câu hỏi nghiên cứu:**
1. Có những người dùng nào có hành vi đánh giá lạ bất thường so với đám đông không?
2. Những bộ phim nào gây ra ý kiến trái chiều mạnh nhất (người thích + kẻ ghét)?
3. Isolation Forest và LOF có đồng thuận với nhau không?

## Luồng xử lý

```
user_features_train.parquet  ──► Isolation Forest + LOF ──► User Anomaly Scores
movie_features_train.parquet ──► Robust Std Z-Score     ──► Movie Polarization Scores
        │
        ▼
  Visualizations (Scatter, Histogram, Score Distribution)
        │
        ▼
  Evaluation (IF vs LOF Agreement, Jaccard, Correlation)
        │
        ▼
  Case Studies + Export Artifacts → artifacts/story_C/
```

## 0. Imports & Setup

In [1]:
import os
import json
import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR    = 'data-warehousing'
STORY_C_DIR = os.path.join('artifacts', 'story_C')
TABLES_OUT  = os.path.join(STORY_C_DIR, 'tables')
REPORTS_OUT = os.path.join(STORY_C_DIR, 'reports')
FIGURES_OUT = os.path.join(STORY_C_DIR, 'figures')

for d in [TABLES_OUT, REPORTS_OUT, FIGURES_OUT]:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
print('✅ Setup OK')

✅ Setup OK


## 1. Load Data

In [2]:
paths = {
    'user_features':  os.path.join(DATA_DIR, 'user_features_train.parquet'),
    'movie_features': os.path.join(DATA_DIR, 'movie_features_train.parquet'),
    'interactions':   os.path.join(DATA_DIR, 'interactions_train.parquet'),
    'movies':         os.path.join(DATA_DIR, 'dim_movies_clean.parquet'),
}

for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing input: {path}')

df_user    = pd.read_parquet(paths['user_features'])
df_movie   = pd.read_parquet(paths['movie_features'])
df_inter   = pd.read_parquet(paths['interactions'])
df_movies  = pd.read_parquet(paths['movies'])

print(f'User features    : {df_user.shape}')
print(f'Movie features   : {df_movie.shape}')
print(f'Interactions     : {df_inter.shape}')
print(f'Movie metadata   : {df_movies.shape}')
df_user.head(3)

User features    : (322397, 29)
Movie features   : (76232, 29)
Interactions     : (30296556, 5)
Movie metadata   : (86537, 5)


,userId,n_ratings,rating_mean,rating_std,rating_min,rating_max,first_dt,last_dt,active_days,n_tag_events,...,genre_pref__film_noir,genre_pref__horror,genre_pref__imax,genre_pref__musical,genre_pref__mystery,genre_pref__romance,genre_pref__sci_fi,genre_pref__thriller,genre_pref__war,genre_pref__western
0,1,55,3.990909,0.754560,2.0,5.0,2008-11-03 17:31:43+00:00,2008-11-03 18:35:15+00:00,1,0,...,0.0,4.166667,3.000000,3.833333,4.00,4.107143,3.500000,3.750000,4.666667,0.00
1,2,81,3.506173,1.073819,1.0,5.0,1996-06-26 18:59:11+00:00,1996-06-26 19:19:26+00:00,1,0,...,0.0,3.500000,4.333333,3.500000,3.25,3.666667,2.666667,3.428571,4.500000,3.75
2,3,27,4.888889,0.423659,3.0,5.0,2018-09-05 18:57:20+00:00,2018-09-05 19:04:04+00:00,1,0,...,0.0,5.000000,5.000000,0.000000,5.00,5.000000,4.400000,4.875000,5.000000,5.00


## 2. Preprocessing — User Features

In [3]:
exclude = [c for c in df_user.columns
           if c in ('userId', 'first_dt', 'last_dt') or df_user[c].dtype == 'object']
feat_cols = [c for c in df_user.columns if c not in exclude]

X = df_user[feat_cols].fillna(0).copy()

# Log-transform count columns
for c in [c for c in feat_cols if 'n_ratings' in c or 'count' in c]:
    X[c] = np.log1p(X[c])

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feat_cols, index=df_user.index)

print(f'Feature matrix: {X_scaled.shape}')
print(f'Feature columns: {feat_cols}')

Feature matrix: (322397, 26)
Feature columns: ['n_ratings', 'rating_mean', 'rating_std', 'rating_min', 'rating_max', 'active_days', 'n_tag_events', 'genre_pref__action', 'genre_pref__adventure', 'genre_pref__animation', 'genre_pref__children', 'genre_pref__comedy', 'genre_pref__crime', 'genre_pref__documentary', 'genre_pref__drama', 'genre_pref__fantasy', 'genre_pref__film_noir', 'genre_pref__horror', 'genre_pref__imax', 'genre_pref__musical', 'genre_pref__mystery', 'genre_pref__romance', 'genre_pref__sci_fi', 'genre_pref__thriller', 'genre_pref__war', 'genre_pref__western']


## 3. Phát Hiện User Bất Thường

### Phương pháp

| Model | Cách hoạt động | Ưu điểm |
|-------|---------------|--------|
| **Isolation Forest** | Tách ngẫu nhiên data points — outlier bị tách nhanh hơn | Nhanh, scale tốt |
| **LOF** (Local Outlier Factor) | So sánh mật độ local của mỗi điểm với hàng xóm | Bắt được local outlier |

**Combined Score** = trung bình rank chuẩn hóa của cả 2 model (ensemble approach).

In [4]:
SAMPLE_SIZE = min(30_000, len(X_scaled))
rng = np.random.RandomState(RANDOM_STATE)
idx = rng.choice(len(X_scaled), SAMPLE_SIZE, replace=False)

X_sample  = X_scaled.iloc[idx]
user_ids  = df_user['userId'].iloc[idx].values

print(f'Running anomaly detection on {SAMPLE_SIZE:,} users...')

# ── Isolation Forest ──────────────────────────────────────────────────────────
iso = IsolationForest(n_estimators=200, contamination=0.05,
                      random_state=RANDOM_STATE, n_jobs=-1)
if_labels = iso.fit_predict(X_sample)      # -1 = anomaly, 1 = normal
if_raw    = -iso.score_samples(X_sample)   # higher = more anomalous
print(f'  IF anomalies detected: {(if_labels == -1).sum():,}')

# ── LOF ───────────────────────────────────────────────────────────────────────
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
lof_labels = lof.fit_predict(X_sample)
lof_raw    = -lof.negative_outlier_factor_
print(f'  LOF anomalies detected: {(lof_labels == -1).sum():,}')

# ── Combined Score ─────────────────────────────────────────────────────────────
def normalise(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

combined = 0.5 * normalise(if_raw) + 0.5 * normalise(lof_raw)

user_scores = pd.DataFrame({
    'userId':           user_ids,
    'iso_forest_score': if_raw,
    'iso_forest_label': if_labels,
    'lof_score':        lof_raw,
    'lof_label':        lof_labels,
    'combined_score':   combined,
    'method':           'isolation_forest+lof',
})
user_scores['rank'] = user_scores['combined_score'].rank(
    ascending=False, method='min').astype(int)
user_scores.sort_values('rank', inplace=True)
user_scores.reset_index(drop=True, inplace=True)

print(f'\nTop 5 most anomalous users:')
user_scores[['userId','rank','combined_score','iso_forest_score','lof_score']].head()

Running anomaly detection on 30,000 users...
  IF anomalies detected: 1,500
  LOF anomalies detected: 1,500

Top 5 most anomalous users:


,userId,rank,combined_score,iso_forest_score,lof_score
0,273882,1,0.856837,0.573812,2.846449e+10
1,88390,2,0.500000,0.651155,1.266995e+00
2,302207,3,0.486219,0.562144,8.595241e+09
3,293543,4,0.479674,0.640174,1.256735e+00
4,99683,5,0.476777,0.573060,6.907348e+09


## 4. Phát Hiện Phim Gây Phân Cực

**Phân cực** = phim có độ lệch chuẩn rating cao — người xem hoặc rất thích hoặc rất ghét.  
Dùng **Robust Z-Score** trên `rating_std` để chuẩn hóa và so sánh giữa các phim.

In [5]:
df = df_movie[['movieId'] + [c for c in df_movie.columns if c != 'movieId']].copy()

# Robust Z-Score trên rating_std
if 'rating_std' in df.columns:
    med = df['rating_std'].median()
    mad = np.abs(df['rating_std'] - med).median()
    df['std_zscore'] = (df['rating_std'] - med) / (mad * 1.4826 + 1e-9)
else:
    df['std_zscore'] = 0.0
    print('⚠️  rating_std not found — polarization scores will be zero')

# Lọc phim có đủ số lượng rating (tránh phim chỉ 1-2 người xem)
if 'n_ratings' in df.columns:
    min_count = max(50, df['n_ratings'].quantile(0.5))
    df_active = df[df['n_ratings'] >= min_count].copy()
    print(f'Films with >= {min_count:.0f} ratings: {len(df_active):,} / {len(df):,}')
else:
    df_active = df.copy()

df_active['polarization_score'] = df_active['std_zscore'].clip(lower=0)
df_active['rank'] = df_active['polarization_score'].rank(ascending=False, method='min').astype(int)
df_active.sort_values('rank', inplace=True)

# Join titles
movie_meta = df_movies[['movieId', 'title', 'genres']]
df_active  = df_active.merge(movie_meta, on='movieId', how='left')

keep_cols = ['movieId', 'rank', 'polarization_score']
for c in ['title', 'genres', 'rating_mean', 'rating_std', 'n_ratings']:
    if c in df_active.columns:
        keep_cols.append(c)
df_active['method'] = 'robust_std_rank'
movie_scores = df_active[keep_cols + ['method']].reset_index(drop=True)

print(f'\nTop 10 most polarizing movies:')
movie_scores[['rank', 'title', 'rating_mean', 'rating_std', 'n_ratings', 'polarization_score']].head(10)

Films with >= 50 ratings: 14,874 / 76,232

Top 10 most polarizing movies:


,rank,title,rating_mean,rating_std,n_ratings,polarization_score
0,1,Fateful Findings (2013),2.750000,1.821236,72,1.858583
1,2,Santa with Muscles (1996),2.433099,1.672927,142,1.568987
2,3,"Room, The (2003)",2.495150,1.658598,1031,1.541007
3,4,Lotto Land (1995),3.321429,1.638815,56,1.502378
4,5,God's Not Dead 2 (2016),2.386792,1.628005,53,1.481269
5,6,After We Collided (2020),2.698276,1.624753,58,1.474920
6,7,God's Not Dead (2014),2.361538,1.621794,195,1.469141
7,8,Expelled: No Intelligence Allowed (2008),2.174497,1.606497,149,1.439272
8,9,Barbie and the Three Musketeers (2009),3.090909,1.597857,66,1.422402
9,10,The Kissing Booth 3 (2021),2.570000,1.590822,50,1.408664


## 5. Visualizations

### 5.1 User Anomaly Scatter — IF Score vs LOF Score

In [6]:
df_plot = user_scores.copy()
df_plot['label'] = df_plot['iso_forest_label'].map({-1: 'Anomaly', 1: 'Normal'})

fig = px.scatter(
    df_plot.head(5000),
    x='iso_forest_score', y='lof_score',
    color='label', opacity=0.6,
    color_discrete_map={'Anomaly': '#FF4500', 'Normal': '#7EC8E3'},
    title='User Anomaly: Isolation Forest Score vs LOF Score',
    labels={
        'iso_forest_score': 'Isolation Forest Score (cao = bất thường hơn)',
        'lof_score':        'LOF Score (cao = bất thường hơn)',
    },
)
fig.update_layout(
    width=800, height=550,
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(family='Inter, sans-serif', size=12),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.write_html(os.path.join(FIGURES_OUT, 'user_anomaly_scatter.html'))
fig.show()
print('Saved user_anomaly_scatter.html')

Saved user_anomaly_scatter.html


### 5.2 Movie Polarization Histogram

In [7]:
fig = px.histogram(
    movie_scores, x='polarization_score', nbins=60,
    title='Phân bố Polarization Score (Robust Std Z-Score)',
    labels={'polarization_score': 'Polarization Score'},
    color_discrete_sequence=['#7EC8E3'],
)
# Đánh dấu top 5 phim phân cực nhất
for _, row in movie_scores.head(5).iterrows():
    label = str(row.get('title', row['movieId']))[:25]
    fig.add_vline(x=row['polarization_score'], line_dash='dash', line_color='#FFD700')
    fig.add_annotation(
        x=row['polarization_score'], y=0,
        text=label, showarrow=True, arrowhead=2,
        font=dict(color='#FFD700', size=9), yshift=10,
    )
fig.update_layout(
    width=800, height=450,
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(family='Inter, sans-serif', size=12),
)
fig.write_html(os.path.join(FIGURES_OUT, 'movie_polarization_hist.html'))
fig.show()
print('Saved movie_polarization_hist.html')

Saved movie_polarization_hist.html


## 6. Đánh Giá Mô Hình (Evaluation)

### 6.1 Model Agreement — IF và LOF có đồng thuận không?

Vì không có nhãn ground truth, ta đánh giá chất lượng bằng **consistency** giữa 2 model.  
Dùng **Jaccard Similarity** và **Score Correlation**.

In [8]:
if_anomalies  = set(user_scores[user_scores['iso_forest_label'] == -1]['userId'])
lof_anomalies = set(user_scores[user_scores['lof_label'] == -1]['userId'])
intersection  = if_anomalies & lof_anomalies
union_set     = if_anomalies | lof_anomalies
jaccard       = len(intersection) / len(union_set) if union_set else 0
corr          = user_scores['iso_forest_score'].corr(user_scores['lof_score'])

print('=' * 50)
print(f'  IF flagged     : {len(if_anomalies):,}')
print(f'  LOF flagged    : {len(lof_anomalies):,}')
print(f'  Both flagged   : {len(intersection):,}')
print(f'  Jaccard        : {jaccard:.3f}  (higher = more agreement)')
print(f'  Score Corr     : {corr:.3f}  (Pearson)')
print('=' * 50)

if jaccard > 0.3:
    print('→ Strong agreement: both models identify similar outliers.')
elif jaccard > 0.15:
    print('→ Moderate agreement: IF catches global outliers, LOF catches local density deviations.')
else:
    print('→ Low agreement: the two models are detecting different kinds of outliers.')

  IF flagged     : 1,500
  LOF flagged    : 1,500
  Both flagged   : 211
  Jaccard        : 0.076  (higher = more agreement)
  Score Corr     : 0.042  (Pearson)
→ Low agreement: the two models are detecting different kinds of outliers.


In [9]:
plt.figure(figsize=(8, 6))
sample_plot = user_scores.sample(min(2000, len(user_scores)), random_state=RANDOM_STATE)
colors = sample_plot['iso_forest_label'].map({-1: '#FF4500', 1: '#7EC8E3'})
plt.scatter(sample_plot['iso_forest_score'], sample_plot['lof_score'],
            c=colors, alpha=0.4, s=15)
# Regression line
m, b = np.polyfit(sample_plot['iso_forest_score'], sample_plot['lof_score'], 1)
xs = np.linspace(sample_plot['iso_forest_score'].min(), sample_plot['iso_forest_score'].max(), 100)
plt.plot(xs, m*xs+b, color='red', linewidth=2, label=f'r = {corr:.3f}')
plt.title('IF Score vs LOF Score — Agreement Analysis')
plt.xlabel('Isolation Forest Score')
plt.ylabel('LOF Score')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_OUT, 'eval_if_lof_agreement.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved eval_if_lof_agreement.png')

Saved eval_if_lof_agreement.png


### 6.2 Score Distribution — Combined Anomaly Score

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Combined Anomaly Score Distribution', fontsize=13, fontweight='bold')

sns.histplot(user_scores['combined_score'], bins=50, kde=True,
             ax=axes[0], color='mediumpurple')
axes[0].set_title('Histogram + KDE')
axes[0].set_xlabel('Combined Score')
axes[0].axvline(user_scores['combined_score'].quantile(0.95),
                color='red', linestyle='--', label='95th percentile')
axes[0].legend(); axes[0].grid(alpha=0.3)

sns.boxplot(x=user_scores['combined_score'], ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot')
axes[1].set_xlabel('Combined Score')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_OUT, 'eval_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved eval_score_distribution.png')

Saved eval_score_distribution.png


### 6.3 Movie Polarization — Mean vs Std Dev

In [11]:
if 'rating_mean' in movie_scores.columns and 'rating_std' in movie_scores.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=movie_scores, x='rating_mean', y='rating_std',
                    hue='polarization_score', palette='viridis', alpha=0.6, s=30)
    plt.title('Movie Mean Rating vs Std Dev\n(màu = mức độ phân cực)')
    plt.xlabel('Average Rating')
    plt.ylabel('Standard Deviation của Rating')
    # Label top 5
    for i in range(min(5, len(movie_scores))):
        row = movie_scores.iloc[i]
        title_short = str(row.get('title', row['movieId']))[:20]
        plt.text(row['rating_mean']+0.03, row['rating_std'],
                 title_short, fontsize=8, fontweight='bold', color='darkred')
    plt.colorbar = plt.gcf().axes[-1]
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_OUT, 'eval_movie_mean_vs_std.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved eval_movie_mean_vs_std.png')
else:
    print('⚠️ rating_mean or rating_std not available in movie_scores')

Saved eval_movie_mean_vs_std.png


### 6.4 Bảng Tóm Tắt Metrics

In [12]:
eval_data = {
    'Total Users Analyzed':         len(user_scores),
    'IF Anomaly Count':              len(if_anomalies),
    'LOF Anomaly Count':             len(lof_anomalies),
    'Both Models Agree (Anomaly)':   len(intersection),
    'Jaccard Similarity':            round(jaccard, 4),
    'IF-LOF Score Correlation (r)':  round(float(corr), 4),
    'Mean Combined Score':           round(float(user_scores['combined_score'].mean()), 4),
    'P95 Combined Score':            round(float(user_scores['combined_score'].quantile(0.95)), 4),
    'Total Movies Analyzed':         len(movie_scores),
    'Max Polarization Score':        round(float(movie_scores['polarization_score'].max()), 4),
}

df_eval = pd.DataFrame(eval_data.items(), columns=['Metric', 'Value'])
df_eval.to_csv(os.path.join(REPORTS_OUT, 'eval_anomaly_metrics.csv'), index=False)
print('Saved eval_anomaly_metrics.csv')
df_eval

Saved eval_anomaly_metrics.csv


,Metric,Value
0,Total Users Analyzed,30000.0000
1,IF Anomaly Count,1500.0000
2,LOF Anomaly Count,1500.0000
3,Both Models Agree (Anomaly),211.0000
4,Jaccard Similarity,0.0757
5,IF-LOF Score Correlation (r),0.0419
6,Mean Combined Score,0.1586
7,P95 Combined Score,0.3267
8,Total Movies Analyzed,14874.0000
9,Max Polarization Score,1.8586


## 7. Export Artifacts

In [13]:
# ── 7.1 Anomaly Score Tables ──────────────────────────────────────────────────
user_scores.to_parquet(os.path.join(TABLES_OUT, 'user_anomaly_scores.parquet'), index=False)
print(f'Saved user_anomaly_scores.parquet  ({len(user_scores):,} users)')

movie_scores.to_parquet(os.path.join(TABLES_OUT, 'movie_anomaly_scores.parquet'), index=False)
print(f'Saved movie_anomaly_scores.parquet ({len(movie_scores):,} movies)')

# ── 7.2 Case Studies Report ───────────────────────────────────────────────────
lines = [
    '# Story C: Behavioral Weirdness — Case Studies\n',
    '_Auto-generated. Mô tả chi tiết top users bất thường và phim gây phân cực._\n',
    '## 🔴 Top Anomalous Users\n',
]
for _, row in user_scores.head(5).iterrows():
    uid = row['userId']
    u_ints = df_inter[df_inter['userId'] == uid] if df_inter is not None else pd.DataFrame()
    lines.append(f'### User {uid} (Rank #{int(row["rank"])})')
    lines.append(f'- **Combined Score:** {row["combined_score"]:.4f}')
    lines.append(f'- **Isolation Forest:** {row["iso_forest_score"]:.4f}')
    lines.append(f'- **LOF Score:** {row["lof_score"]:.4f}')
    if len(u_ints) > 0:
        lines.append(f'- **Ratings count:** {len(u_ints)}')
        lines.append(f'- **Mean rating:** {u_ints["rating"].mean():.2f} | Std: {u_ints["rating"].std():.2f}')
    lines.append('---\n')

lines += [
    '## 🎬 Top Polarizing Movies\n',
    '| Rank | Title | Mean | Std | # Ratings | Polarization Score |',
    '|------|-------|------|-----|-----------|-------------------|',
]
for _, row in movie_scores.head(10).iterrows():
    title  = str(row.get('title',  row['movieId']))[:40]
    mean_r = f"{row['rating_mean']:.2f}" if 'rating_mean' in row and not pd.isna(row.get('rating_mean')) else '?'
    std_r  = f"{row['rating_std']:.2f}"  if 'rating_std'  in row and not pd.isna(row.get('rating_std'))  else '?'
    n_r    = int(row['n_ratings'])        if 'n_ratings'   in row and not pd.isna(row.get('n_ratings'))   else '?'
    lines.append(f"| {row['rank']} | {title} | {mean_r} | {std_r} | {n_r} | {row['polarization_score']:.4f} |")

with open(os.path.join(REPORTS_OUT, 'case_studies.md'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print('Saved case_studies.md')

# ── 7.3 Summary ───────────────────────────────────────────────────────────────
summary = f"""# Story C: Behavioral Weirdness — Summary

Executed: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Overview
Anomaly detection trên {len(user_scores):,} users từ tập train. Polarization analysis trên {len(movie_scores):,} phim.

## User Anomaly Detection
- **Method:** Isolation Forest + LOF (contamination=5%, combined normalized score)
- **IF anomalies:** {len(if_anomalies):,} ({len(if_anomalies)/len(user_scores)*100:.1f}%)
- **LOF anomalies:** {len(lof_anomalies):,} ({len(lof_anomalies)/len(user_scores)*100:.1f}%)
- **Model agreement (Jaccard):** {jaccard:.3f}
- **Score correlation:** {corr:.3f}

## Movie Polarization
- **Method:** Robust Z-Score trên rating_std (sau khi lọc min_count)
- Top phim phân cực là các controversial titles — vừa được yêu thích vừa bị ghét bỏ.
"""
with open(os.path.join(REPORTS_OUT, 'summary.md'), 'w', encoding='utf-8') as f:
    f.write(summary)
print('Saved summary.md')

# ── 7.4 Run Manifest ─────────────────────────────────────────────────────────
manifest = {
    'story':      'Story C: Behavioral Weirdness',
    'timestamp':  datetime.datetime.now().isoformat(),
    'notebook':   'story_c_behavioral_weirdness.ipynb',
    'inputs':     ['user_features_train.parquet', 'movie_features_train.parquet',
                   'interactions_train.parquet', 'dim_movies_clean.parquet'],
    'methods':    {'user_anomaly': ['IsolationForest', 'LocalOutlierFactor'],
                   'movie_polarization': ['robust_std_rank']},
    'parameters': {'iso_forest': {'n_estimators': 200, 'contamination': 0.05},
                   'lof':        {'n_neighbors': 20, 'contamination': 0.05},
                   'sample_size': SAMPLE_SIZE},
    'metrics':    {'n_users_sampled': int(len(user_scores)),
                   'n_anomalous_iso': int(len(if_anomalies)),
                   'n_anomalous_lof': int(len(lof_anomalies)),
                   'jaccard':         round(jaccard, 4),
                   'score_corr':      round(float(corr), 4),
                   'n_movies_analyzed': int(len(movie_scores))},
}
with open(os.path.join(REPORTS_OUT, 'run_manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=4)
print('Saved run_manifest.json')

print('\n✅ All artifacts exported to', STORY_C_DIR)

Saved user_anomaly_scores.parquet  (30,000 users)
Saved movie_anomaly_scores.parquet (14,874 movies)
Saved case_studies.md
Saved summary.md
Saved run_manifest.json

✅ All artifacts exported to artifacts\story_C


## 8. Kết Luận & Diễn Giải

### Về phát hiện User bất thường

- **Isolation Forest** bắt được **global outliers** — những user có profile hoàn toàn khác xa phần còn lại.
- **LOF** bắt được **local density anomalies** — những user lạ so với nhóm hàng xóm gần nhất của họ.
- **Jaccard > 0.3**: hai model khá nhất quán → outliers là thật sự bất thường về nhiều mặt.
- **Jaccard < 0.15**: hai model phát hiện các loại outlier khác nhau → dùng combined score là cần thiết.

### Về phim phân cực

- Phim có `polarization_score` cao = rating_std cao so với median → khán giả phân chia ý kiến mạnh.
- Thường là: **cult films**, **art-house**, **horror**, **thể loại gây tranh cãi**.
- Lọc `n_ratings >= 50` tránh nhiễu từ phim ít người xem.

### Hướng cải thiện

1. **UMAP** để visualize không gian user feature trước khi chạy LOF — LOF nhạy với chiều cao (curse of dimensionality)
2. **Autoencoder** reconstruction error — outlier = không thể reconstruct tốt
3. **Temporal analysis** — kiểm tra xem các anomalous users có rating tập trung vào thời điểm bất thường không (bot-like patterns)